In [4]:
"""
spid_v4_algorithm.py
====================
SPID v4 — Spectral Picard-Ishikawa-Dykstra for the Nearest Correlation Matrix.

Reference:
  Ibrahim, O.E. (2026). Accelerated Computation of the Nearest Correlation
  Matrix via Spectral Picard-Ishikawa-Dykstra (SPID) Iterations.

Algorithm 1 (paper, Section 4.6):
  Steps [1] Dykstra Cycle
        [2] Ishikawa Nesting
        [3] Alternating BB Step Size  (Dai & Yuan 2002)
        [4] Update
        [5] Stopping Criterion
        [6] Shift
        Post-processing: feasibility polish

Usage:
  python spid_v4_algorithm.py
"""

import numpy as np
import scipy.linalg as la
import pandas as pd
import time
import tracemalloc
from collections import defaultdict


# ======================================================================
# SECTION 1 — CORE PROJECTION OPERATORS
# ======================================================================

def proj_U(A):
    """
    P_U(A): project onto unit-diagonal subspace U^n.
    Sets all diagonal entries to 1.  O(n).
    Satisfies: diag(P_U(A)) = 1 for any A.
    """
    X = A.copy()
    np.fill_diagonal(X, 1.0)
    return X


def proj_S(A):
    """
    P_S(A): project onto positive semidefinite cone S^+_n.
    Computes the eigendecomposition A = Q Λ Q^T and returns Q Λ+ Q^T,
    where Λ+ clips all negative eigenvalues to zero.  O(n^3).

    Uses subset_by_value=(0, inf) to return only non-negative eigenpairs,
    exploiting LAPACK dsyevr for efficiency on large, nearly-PSD matrices.
    """
    try:
        w, V = la.eigh(A, subset_by_value=(0.0, np.inf))
        X = V @ np.diag(w) @ V.T
    except ValueError:
        # subset_by_value fails when no non-negative eigenvalues exist
        X = np.zeros_like(A)
    return (X + X.T) / 2.0          # symmetrise to kill floating-point drift


# ======================================================================
# SECTION 2 — TIMER UTILITY  (Julia TimerOutputs style)
# ======================================================================

class SectionTimer:
    """
    Accumulates wall-time and peak memory per named section.
    Prints a Julia-style profiling report.
    """

    def __init__(self):
        self.reset()

    def reset(self):
        self.sections    = defaultdict(lambda: {"ncalls": 0, "time": 0.0, "alloc": 0.0})
        self.total_time  = 0.0
        self.total_alloc = 0.0

    def record(self, name, dt, alloc_mb):
        self.sections[name]["ncalls"] += 1
        self.sections[name]["time"]   += dt
        self.sections[name]["alloc"]  += alloc_mb

    def print_report(self, title, iters, min_eig, frob_err):
        tot_t = self.total_time
        tot_a = self.total_alloc
        sep   = "=" * 80
        sep2  = "-" * 80
        print(f"\n{sep}")
        print(f"=== {title} ===")
        print(f"Converged after {iters} iterations")
        print(sep2)
        print(f"{'':30s}  {'Time':>22s}   {'Allocations':>22s}")
        print(f"{'Tot / % measured:':<30s}  "
              f"{self._fmt_time(tot_t):>10s} / {'100.0%':>6s}   "
              f"{self._fmt_mem(tot_a):>10s} / {'100.0%':>6s}")
        print(sep2)
        print(f"{'Section':<30s}  {'ncalls':>6s}  {'time':>9s}  {'%tot':>6s}  "
              f"{'avg':>9s}  {'alloc':>9s}  {'avg':>9s}")
        print(sep2)
        for name, v in self.sections.items():
            pct_t = 100 * v["time"]  / tot_t if tot_t > 0 else 0
            avg_t = v["time"]  / v["ncalls"] if v["ncalls"] > 0 else 0
            avg_a = v["alloc"] / v["ncalls"] if v["ncalls"] > 0 else 0
            print(f"  {name:<28s}  {v['ncalls']:>6d}  "
                  f"{self._fmt_time(v['time']):>9s}  {pct_t:>5.1f}%  "
                  f"{self._fmt_time(avg_t):>9s}  "
                  f"{self._fmt_mem(v['alloc']):>9s}  "
                  f"{self._fmt_mem(avg_a):>9s}")
        print(sep2)
        print(f"Minimum Eigenvalue (post-polish): {min_eig:.6e}")
        print(f"|| X - G ||_F:                   {frob_err:.6f}")
        print(sep)

    @staticmethod
    def _fmt_time(t):
        if t >= 1.0:    return f"{t:.2f}s"
        if t >= 1e-3:   return f"{t*1e3:.2f}ms"
        if t >= 1e-6:   return f"{t*1e6:.2f}μs"
        return                  f"{t*1e9:.2f}ns"

    @staticmethod
    def _fmt_mem(m):
        if m >= 1024:   return f"{m/1024:.2f}GiB"
        if m >= 1.0:    return f"{m:.2f}MiB"
        if m >= 1/1024: return f"{m*1024:.2f}KiB"
        return                  f"{m*1024*1024:.2f}B" if m > 0 else "0.00B"


# ======================================================================
# SECTION 3 — SPID v4   (Algorithm 1 in paper)
# ======================================================================

def spid_v4(A, beta=0.5, lambda_max=5.0, alpha_min=0.1,
            tol=1e-5, max_iter=2000, verbose=False):
    """
    SPID v4: Nearest Correlation Matrix via Spectral Picard-Ishikawa-Dykstra.

    Parameters
    ----------
    A          : (n, n) ndarray — input symmetric unit-diagonal matrix G
    beta       : float   — Ishikawa nesting weight, β ∈ (0,1).   Default 0.5
    lambda_max : float   — ABB safeguard ceiling λ_max.           Default 5.0
    alpha_min  : float   — ABB safeguard floor.                   Default 0.1
    tol        : float   — convergence tolerance.                 Default 1e-5
    max_iter   : int     — iteration budget.                      Default 2000
    verbose    : bool    — print per-iteration residuals.         Default False

    Returns
    -------
    X_hat  : (n, n) ndarray — nearest correlation matrix (X ≥ 0, diag = 1)
    iters  : int            — number of iterations taken
    tm     : SectionTimer   — profiling data (call tm.print_report(...) to display)

    Notes
    -----
    Memory layout (4 auxiliary n×n matrices beyond Dykstra state):
      X_prev  — previous iterate (for BB step numerator s_k)
      R_prev  — previous residual direction (for BB step denominator y_k)
      dS, dU  — Dykstra dual correction arrays

    Stopping criterion:
      n < 500 : feasibility proxy  max(||X_next - S||_F, max_i|X_{ii}-1|) < tol
                reuses S from the Dykstra EVD at zero extra cost
      n ≥ 500 : step-norm  ||X_next - X_k||_F < tol
    """

    tm   = SectionTimer()
    n    = A.shape[0]
    X    = A.copy()
    dS   = np.zeros((n, n))
    dU   = np.zeros((n, n))

    alpha_max      = lambda_max / beta   # safeguard ceiling for α_k
    alpha_fallback = 1.0 / beta          # used for k=1 or degenerate denominators
    large_n        = (n >= 500)          # selects stopping criterion branch

    X_prev = X.copy()
    R_prev = np.zeros((n, n))

    t_total0 = time.perf_counter()

    for k in range(1, max_iter + 1):

        # ── [1] Dykstra Cycle ──────────────────────────────────────
        tracemalloc.start()
        t0 = time.perf_counter()

        R     = X + dS
        S     = proj_S(R)          # partial EVD — dominant O(n^3) cost
        dS_new = R - S

        R_U    = S + dU
        W      = proj_U(R_U)       # set diagonal to 1, O(n)
        dU_new = R_U - W
        T_D    = W                 # Dykstra output

        dt = time.perf_counter() - t0
        _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
        tm.record("Project_onto_PSD", dt, peak / 1024**2)

        # ── [2] Ishikawa Nesting ───────────────────────────────────
        tracemalloc.start()
        t0 = time.perf_counter()

        Z_X   = (1 - beta) * X   + beta * T_D
        Z_dS  = (1 - beta) * dS  + beta * dS_new
        Z_dU  = (1 - beta) * dU  + beta * dU_new
        R_curr = X - Z_X          # current residual direction for BB step

        dt = time.perf_counter() - t0
        _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
        tm.record("Ishikawa_Nesting", dt, peak / 1024**2)

        # ── [3] Alternating BB Step Size (Dai & Yuan 2002) ─────────
        tracemalloc.start()
        t0 = time.perf_counter()

        if k == 1:
            alpha = alpha_fallback      # warm-up: no previous iterate
        else:
            s      = X - X_prev         # displacement in X
            y      = R_curr - R_prev    # displacement in residual direction
            tr_SS  = np.sum(s * s)      # ||s||^2_F
            tr_SY  = np.sum(s * y)      # <s, y>_F
            tr_YY  = np.sum(y * y)      # ||y||^2_F

            if k % 2 == 1:              # BB1 on odd iterations
                alpha = tr_SS / tr_SY if tr_SY > 1e-12 else alpha_fallback
            else:                       # BB2 on even iterations
                alpha = tr_SY / tr_YY if tr_YY > 1e-12 else alpha_fallback

            alpha = float(np.clip(alpha, alpha_min, alpha_max))

        dt = time.perf_counter() - t0
        _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
        tm.record("BB_StepSize", dt, peak / 1024**2)

        # ── [4] Update ────────────────────────────────────────────
        tracemalloc.start()
        t0 = time.perf_counter()

        X_next   = X   + alpha * (Z_X   - X)
        dS_next  = dS  + alpha * (Z_dS  - dS)
        dU_next  = dU  + alpha * (Z_dU  - dU)

        # ── [5] Stopping Criterion ────────────────────────────────
        if large_n:
            # Step-norm stopping: cheap, no extra EVD
            converged = la.norm(X_next - X, 'fro') < tol
        else:
            # Feasibility proxy: reuses S from step [1] at zero EVD cost
            psd_res   = la.norm(X_next - S, 'fro')
            diag_res  = np.max(np.abs(np.diag(X_next) - 1.0))
            converged = max(psd_res, diag_res) < tol

        dt = time.perf_counter() - t0
        _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
        tm.record("Update_and_Check", dt, peak / 1024**2)

        if verbose:
            res = la.norm(X_next - X, 'fro')
            print(f"  iter {k:4d}  alpha={alpha:.4f}  res={res:.6e}")

        if converged:
            X = X_next
            break

        # ── [6] Shift ─────────────────────────────────────────────
        X_prev = X.copy()
        R_prev = R_curr.copy()
        X, dS, dU = X_next, dS_next, dU_next

    # Post-processing: feasibility polish
    # Guarantees X_hat ≥ 0 and diag(X_hat) = 1 exactly by construction.
    X_hat = proj_U(proj_S(X))

    tm.total_time  = time.perf_counter() - t_total0
    tm.total_alloc = sum(v["alloc"] for v in tm.sections.values())

    return X_hat, k, tm


# ======================================================================
# SECTION 4 — BASELINE SOLVERS  (for comparison / Table 2 reproduction)
# ======================================================================

def higham_dykstra(A, tol=1e-5, max_iter=2000):
    """
    Plain Dykstra alternating projection (Higham 2002).
    Returns (X, iters, SectionTimer).
    """
    tm = SectionTimer()
    X  = A.copy()
    dS = np.zeros_like(A)
    dU = np.zeros_like(A)

    t0 = time.perf_counter()
    for k in range(1, max_iter + 1):
        tracemalloc.start()
        t1 = time.perf_counter()
        R = X + dS; S = proj_S(R); dS_new = R - S
        dt = time.perf_counter() - t1
        _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
        tm.record("Project_onto_PSD", dt, peak / 1024**2)

        tracemalloc.start()
        t1 = time.perf_counter()
        R_U = S + dU; W = proj_U(R_U); dU_new = R_U - W
        dt = time.perf_counter() - t1
        _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
        tm.record("project_onto_UD!", dt, peak / 1024**2)

        if la.norm(W - X, 'fro') < tol:
            X = W; dS = dS_new; dU = dU_new; break
        X, dS, dU = W, dS_new, dU_new

    tm.total_time  = time.perf_counter() - t0
    tm.total_alloc = sum(v["alloc"] for v in tm.sections.values())
    return X, k, tm


def anderson_dykstra(A, m=4, tol=1e-5, max_iter=2000):
    """
    Anderson-accelerated Dykstra (Higham & Strabic 2016), window m=3.
    Returns (X, iters, SectionTimer).
    """
    tm = SectionTimer()
    n  = A.shape[0]
    X  = A.copy()
    dS = np.zeros_like(A)
    dU = np.zeros_like(A)
    X_hist = np.zeros((n * n, m))
    F_hist = np.zeros((n * n, m))
    m_active = 0

    t0 = time.perf_counter()
    for k in range(1, max_iter + 1):
        tracemalloc.start()
        t1 = time.perf_counter()
        R = X + dS; S = proj_S(R); dS_new = R - S
        dt = time.perf_counter() - t1
        _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
        tm.record("Project_onto_PSD", dt, peak / 1024**2)

        tracemalloc.start()
        t1 = time.perf_counter()
        R_U = S + dU; Gx = proj_U(R_U); dU_new = R_U - Gx
        dt = time.perf_counter() - t1
        _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
        tm.record("project_onto_UD!", dt, peak / 1024**2)

        if la.norm(Gx - X, 'fro') < tol:
            X = Gx; break

        tracemalloc.start()
        t1 = time.perf_counter()
        idx = m_active % m
        X_hist[:, idx] = X.flatten()
        F_hist[:, idx] = (Gx - X).flatten()
        m_active = min(m_active + 1, m)
        if m_active == 1:
            X_new = X_hist[:, 0] + F_hist[:, 0]
        else:
            dF = F_hist[:, 1:m_active] - F_hist[:, :m_active - 1]
            dX = X_hist[:, 1:m_active] - X_hist[:, :m_active - 1]
            try:
                gamma, _, _, _ = la.lstsq(dF, F_hist[:, m_active - 1], cond=1e-8)
                X_new = (X_hist[:, m_active - 1] + F_hist[:, m_active - 1]
                         - (dX + dF) @ gamma)
            except la.LinAlgError:
                X_new = Gx.flatten()
        dt = time.perf_counter() - t1
        _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
        tm.record("Anderson_Mixing", dt, peak / 1024**2)

        X, dS, dU = X_new.reshape((n, n)), dS_new, dU_new

    tm.total_time  = time.perf_counter() - t0
    tm.total_alloc = sum(v["alloc"] for v in tm.sections.values())
    return X, k, tm


# ======================================================================
# SECTION 5 — TEST MATRIX GENERATORS  (all 14 benchmark cases)
# ======================================================================

def gen_rebonato_jackel(n=100, seed=42):
    np.random.seed(seed)
    i = np.arange(n).reshape(-1, 1); j = np.arange(n).reshape(1, -1)
    G = np.exp(-0.1 * np.abs(i - j))
    G += 0.1 * np.random.randn(n, n)
    return proj_U((G + G.T) / 2.0)

def gen_turkay():
    return np.array([[ 1.00, -0.55, -0.15, -0.10],
                     [-0.55,  1.00,  0.90,  0.90],
                     [-0.15,  0.90,  1.00,  0.90],
                     [-0.10,  0.90,  0.90,  1.00]])

def gen_bhansali():
    return np.array([[ 1.00, -0.50, -0.30, -0.25, -0.70],
                     [-0.50,  1.00,  0.90,  0.30,  0.70],
                     [-0.30,  0.90,  1.00,  0.25,  0.20],
                     [-0.25,  0.30,  0.25,  1.00,  0.75],
                     [-0.70,  0.70,  0.20,  0.75,  1.00]])

def gen_finger():
    return np.array([[ 1.00,  0.18, -0.13, -0.26,  0.19, -0.25, -0.12],
                     [ 0.18,  1.00,  0.22, -0.14,  0.31,  0.16,  0.09],
                     [-0.13,  0.22,  1.00,  0.06, -0.08,  0.04,  0.04],
                     [-0.26, -0.14,  0.06,  1.00,  0.85,  0.85,  0.85],
                     [ 0.19,  0.31, -0.08,  0.85,  1.00,  0.85,  0.85],
                     [-0.25,  0.16,  0.04,  0.85,  0.85,  1.00,  0.85],
                     [-0.12,  0.09,  0.04,  0.85,  0.85,  0.85,  1.00]])

def gen_higham_strabic(n=500):
    G = np.ones((n, n)) * 0.5; np.fill_diagonal(G, 1.0)
    G[0, n-1] = G[n-1, 0] = -0.9
    return G

def gen_higham_ex31():
    return np.array([[1.0, 1.0, 0.0],
                     [1.0, 1.0, 1.0],
                     [0.0, 1.0, 1.0]])

def gen_higham_ex33(n=100):
    G = np.eye(n) + 1.1 * np.ones((n, n)) / n
    np.fill_diagonal(G, 1.0)
    return G

def gen_qi_sun_55(n=500, seed=42):
    np.random.seed(seed)
    G = np.random.randn(n, n); C = G @ G.T
    d = np.diag(1.0 / np.sqrt(np.diag(C))); C = d @ C @ d
    R = np.random.uniform(-1, 1, (n, n)); R = (R + R.T) / 2.0
    return C + 1.0 * R

def gen_qi_sun_56(n=500, seed=42):
    np.random.seed(seed)
    G = np.random.uniform(-1, 1, (n, n)); G = (G + G.T) / 2.0
    np.fill_diagonal(G, 1.0)
    return G

def gen_minabutdinov(n=500):
    G = np.eye(n) - 10.0 * np.ones((n, n)) / n
    np.fill_diagonal(G, 1.0)
    return G

def gen_grubisic_pietersz(n=100, seed=42):
    np.random.seed(seed); r = 5
    F = np.random.randn(n, r); G = F @ F.T
    d = np.diag(1.0 / np.sqrt(np.diag(G))); G = d @ G @ d
    G += 0.05 * np.random.randn(n, n)
    return proj_U((G + G.T) / 2.0)

def gen_russell_3000(n=1000, seed=42):
    np.random.seed(seed)
    G = np.random.randn(n, 10) @ np.random.randn(10, n)
    G = (G + G.T) / 2.0
    return proj_U(G)

def gen_custom_extreme(n=200, seed=42):
    np.random.seed(seed)
    G = np.random.uniform(-2e4, 2e4, (n, n)); G = (G + G.T) / 2.0
    np.fill_diagonal(G, 1.0)
    return G

def gen_thesis_mcar(n_stocks=550, n_days=1000, missing_rate=0.25, seed=42):
    np.random.seed(seed)
    factors  = np.random.randn(10, n_days)
    loadings = np.random.randn(n_stocks, 10)
    returns  = loadings @ factors + np.random.randn(n_stocks, n_days) * 0.5
    mask     = np.random.rand(n_stocks, n_days) > missing_rate
    observed = np.where(mask, returns, np.nan)
    df       = pd.DataFrame(observed.T)
    G        = df.corr().fillna(0).values
    G        = (G + G.T) / 2.0
    np.fill_diagonal(G, 1.0)
    return G


# ======================================================================
# SECTION 6 — BENCHMARK RUNNER  (reproduces Table 2 in paper)
# ======================================================================

ALL_TESTS = [
    ("1.  Rebonato-Jackel n=100",  gen_rebonato_jackel()),
    ("2.  Turkay et al.   n=4",    gen_turkay()),
    ("3.  Bhansali-Wise   n=5",    gen_bhansali()),
    ("4.  Finger 1997     n=7",    gen_finger()),
    ("5.  Higham-Strabic  n=500",  gen_higham_strabic()),
    ("6.  Higham Ex 3.1   n=3",    gen_higham_ex31()),
    ("7.  Higham Ex 3.3   n=100",  gen_higham_ex33()),
    ("8.  Qi-Sun 5.5      n=500",  gen_qi_sun_55()),
    ("9.  Qi-Sun 5.6      n=500",  gen_qi_sun_56()),
    ("10. Minabutdinov    n=500",  gen_minabutdinov()),
    ("11. Grubisic-P.     n=100",  gen_grubisic_pietersz()),
    ("12. Russell 3000    n=1000", gen_russell_3000()),
    ("13. Custom Extreme  n=200",  gen_custom_extreme()),
    ("14. MCAR Thesis     n=550",  gen_thesis_mcar()),
]

SOLVERS = [
    ("Higham Baseline", higham_dykstra),
    ("Anderson APM",    anderson_dykstra),
    ("SPID v4",         spid_v4),
]


def run_all_benchmarks():
    """Run all 14 test cases with all 3 solvers and print Table 2."""
    hdr = f"{'Problem':<30}  {'H-Its':>6}  {'H-t':>6}  " \
          f"{'AA-Its':>6}  {'AA-t':>6}  " \
          f"{'SP-Its':>6}  {'SP-t':>6}  {'DiagErr':>8}  {'MinEig':>11}"
    print("\n" + "=" * len(hdr))
    print("TABLE 2 — Benchmark Results  (tol=1e-5, max_iter=2000)")
    print("=" * len(hdr))
    print(hdr)
    print("-" * len(hdr))

    for name, G in ALL_TESTS:
        row_data = {}
        for sname, sfn in SOLVERS:
            X, iters, tm = sfn(G)
            row_data[sname] = {
                "iters": iters,
                "time":  tm.total_time,
                "diag":  np.max(np.abs(np.diag(X) - 1.0)),
                "eig":   np.min(la.eigvalsh(X)),
                "dnf":   (iters >= 2000),
            }

        h  = row_data["Higham Baseline"]
        aa = row_data["Anderson APM"]
        sp = row_data["SPID v4"]

        h_its  = "DNF" if h["dnf"]  else f"{h['iters']:>6d}"
        aa_its = "DNF" if aa["dnf"] else f"{aa['iters']:>6d}"
        sp_its = "DNF" if sp["dnf"] else f"{sp['iters']:>6d}"

        print(f"{name:<30}  {h_its:>6}  {h['time']:>6.2f}  "
              f"{aa_its:>6}  {aa['time']:>6.2f}  "
              f"{sp_its:>6}  {sp['time']:>6.2f}  "
              f"{sp['diag']:>8.1e}  {sp['eig']:>+11.4e}")

    print("-" * len(hdr))


if __name__ == "__main__":
    # Quick single-case demonstration
    print("SPID v4 — Quick Demo  (Thesis MCAR, n=550)")
    print("=" * 60)
    G = gen_thesis_mcar()
    print(f"Input:  n=550,  λ_min(G) = {np.min(la.eigvalsh(G)):.4f}")
    X, iters, tm = spid_v4(G)
    tm.print_report(
        title    = "SPID v4 | MCAR Thesis n=550",
        iters    = iters,
        min_eig  = np.min(la.eigvalsh(X)),
        frob_err = la.norm(X - G, 'fro')
    )

    # Uncomment to run full 14-case benchmark (takes ~10 min):
    run_all_benchmarks()


SPID v4 — Quick Demo  (Thesis MCAR, n=550)
Input:  n=550,  λ_min(G) = -0.4553

=== SPID v4 | MCAR Thesis n=550 ===
Converged after 66 iterations
--------------------------------------------------------------------------------
                                                  Time              Allocations
Tot / % measured:                    3.11s / 100.0%      2.54GiB / 100.0%
--------------------------------------------------------------------------------
Section                         ncalls       time    %tot        avg      alloc        avg
--------------------------------------------------------------------------------
  Project_onto_PSD                  66      2.67s   86.1%    40.53ms  914.00MiB   13.85MiB
  Ishikawa_Nesting                  66   120.82ms    3.9%     1.83ms  609.31MiB    9.23MiB
  BB_StepSize                       66    78.05ms    2.5%     1.18ms  450.12MiB    6.82MiB
  Update_and_Check                  66   109.87ms    3.5%     1.66ms  628.41MiB    9.52MiB
---